# 온도 원본의 컬럼별 결측 확인

결측은 그대로 두고 시간과 LOT를 확인함. 기록만으로 통신 지연 등 누락 원인을 확정하지 않음.
예측에 쓸 값을 채울 때는 미래값이나 다른 LOT를 사용하지 않도록 별도 기준을 정해야 함.


In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
FIG_DIR = os.path.join(ROOT, "figures")
ENC = "utf-8-sig"

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
BASE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")


file_path = TEMP_CSV
df = pd.read_csv(file_path, encoding=ENC)

print(f"전체 데이터 규모: {df.shape[0]:,}행 × {df.shape[1]}열")

---
# 전체 결측치 요약 현황

In [ ]:
# 결측치가 1개 이상 존재하는 컬럼만 추출하여 비율과 함께 요약
null_summary = pd.DataFrame({
    '결측치_개수': df.isnull().sum(),
    '결측치_비율(%)': (df.isnull().sum() / len(df)) * 100
})

# 결측치가 있는 컬럼만 필터링하여 출력
null_cols = null_summary[null_summary['결측치_개수'] > 0]
print(f"총 {len(null_cols)}개 컬럼에서 결측치 발견")
null_cols

## 총 16개 컬럼에서 결측치 발견
### `컬럼명`을 클릭하여 해당 정보로 이동

| 컬럼 | 결측치 개수 | 컬럼 의미 |
| --- | --- | --- |
| [`CLN-IN`](#desc-cln-in) | 4 | 세정조 입측 온도 (세정 공정 입측 부근 측정 온도) |
| [`TC-Z1`](#desc-tc-z1) | 19 | 건조존 1존 온도 (건조존 1구간 열전대 측정 온도) |
| [`TC-Z2`](#desc-tc-z2) | 23 | 건조존 2존 온도 (건조존 2구간 열전대 측정 온도) |
| [`TC-Z3`](#desc-tc-z3) | 5 | 가열·균열존 1존 온도 (가열·균열존 1구간 측정 온도) |
| [`TC-Z4`](#desc-tc-z4) | 12 | 가열·균열존 2존 온도 (가열·균열존 2구간 측정 온도) |
| [`TC-Z5`](#desc-tc-z5) | 15 | 가열·균열존 3존 온도 (가열·균열존 3구간 측정 온도) |
| [`TC-Z6`](#desc-tc-z6) | 16 | 가열·균열존 4존 온도 (가열·균열존 4구간 측정 온도) |
| [`OP-Z3`](#desc-op-z3) | 254 | 가열·균열존 1존 제어출력 (가열·균열존 1구간 제어 출력 비율) |
| [`CP-Z4M`](#desc-cp-z4m) | 13 | 카본포텐셜 모니터 (가열·균열존 분위기 가스 모니터링 값) |
| [`TC-Z7`](#desc-tc-z7) | 22 | 과시효대 1존 온도 (과시효대 1구간 측정 온도) |
| [`TC-Z8`](#desc-tc-z8) | 17 | 과시효대 2존 온도 (과시효대 2구간 측정 온도) |
| [`TC-Z9`](#desc-tc-z9) | 18 | 과시효대 슬러지 제거부 온도 (과시효대 3구간 측정 온도) |
| [`OP-Z7`](#desc-op-z7) | 20 | 과시효대 1존 제어출력 (과시효대 1구간 제어 출력 비율) |
| [`OP-Z8`](#desc-op-z8) | 17 | 과시효대 2존 제어출력 (과시효대 2구간 제어 출력 비율) |
| [`TC-OUT1`](#desc-tc-out1) | 6 | 냉각대 1존 온도 (냉각대 1구간 / 출구 1측 측정 온도) |
| [`TC-OUT2`](#desc-tc-out2) | 13 | 냉각대 2존 온도 (냉각대 2구간 / 출구 2측 측정 온도) |

---

아래는 센서별 수치 요약, 결측 발생 시각, LOT별 평균과 범위 순서임.


---
# <span id="desc-cln-in"></span>`CLN-IN` (세정조 입측 온도)

## 1. CLN-IN 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 평균, 표준편차, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'CLN-IN'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. CLN-IN 결측치 발생 시점 및 LOT 확인
- `CLN-IN` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'CLN-IN'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 CLN-IN 온도 집계
- 작업 단위(LOT)별 세정조 입측 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'CLN-IN'

# LOT별 CLN-IN 온도 집계
cln_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
cln_lot_summary['평균온도'] = cln_lot_summary['평균온도'].round(2)

print(f"총 {len(cln_lot_summary)}개 LOT에 대한 CLN-IN 수치 요약:\n")
for i, row in cln_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:5.2f}°C ~ {row['최고온도']:5.2f}°C ({row['측정건수']:,}건)")

### CLN-IN 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-z1"></span>`TC-Z1` (건조존 1존 온도)

## 1. TC-Z1 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z1'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z1 결측치 발생 시점 및 LOT 확인
- `TC-Z1` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z1'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z1 온도 집계
- 작업 단위(LOT)별 건조존 1존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z1'

# LOT별 TC-Z1 온도 집계
tc1_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc1_lot_summary['평균온도'] = tc1_lot_summary['평균온도'].round(2)

print(f"총 {len(tc1_lot_summary)}개 LOT에 대한 TC-Z1 수치 요약:\n")
for i, row in tc1_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z1 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-z2"></span>`TC-Z2` (건조존 2존 온도)

## 1. TC-Z2 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z2'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z2 결측치 발생 시점 및 LOT 확인
- `TC-Z2` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z2'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z2 온도 집계
- 작업 단위(LOT)별 건조존 2존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z2'

# LOT별 TC-Z2 온도 집계
tc2_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc2_lot_summary['평균온도'] = tc2_lot_summary['평균온도'].round(2)

print(f"총 {len(tc2_lot_summary)}개 LOT에 대한 TC-Z2 수치 요약:\n")
for i, row in tc2_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z2 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-z3"></span>`TC-Z3` (가열·균열존 1존 온도)

## 1. TC-Z3 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z3'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z3 결측치 발생 시점 및 LOT 확인
- `TC-Z3` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z3'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z3 온도 집계
- 작업 단위(LOT)별 3존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z3'

# LOT별 TC-Z3 온도 집계
tc3_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc3_lot_summary['평균온도'] = tc3_lot_summary['평균온도'].round(2)

print(f"총 {len(tc3_lot_summary)}개 LOT에 대한 TC-Z3 수치 요약:\n")
for i, row in tc3_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z3 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-z4"></span>`TC-Z4` (가열·균열존 2존 온도)

## 1. TC-Z4 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z4'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z4 결측치 발생 시점 및 LOT 확인
- `TC-Z4` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z4'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z4 온도 집계
- 작업 단위(LOT)별 4존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z4'

# LOT별 TC-Z4 온도 집계
tc4_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc4_lot_summary['평균온도'] = tc4_lot_summary['평균온도'].round(2)

print(f"총 {len(tc4_lot_summary)}개 LOT에 대한 TC-Z4 수치 요약:\n")
for i, row in tc4_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z4 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-z5"></span>`TC-Z5` (가열·균열존 3존 온도)

## 1. TC-Z5 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z5'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z5 결측치 발생 시점 및 LOT 확인
- `TC-Z5` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z5'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z5 온도 집계
- 작업 단위(LOT)별 5존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z5'

# LOT별 TC-Z5 온도 집계
tc5_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc5_lot_summary['평균온도'] = tc5_lot_summary['평균온도'].round(2)

print(f"총 {len(tc5_lot_summary)}개 LOT에 대한 TC-Z5 수치 요약:\n")
for i, row in tc5_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z5 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-z6"></span>`TC-Z6` (가열·균열존 4존 온도)

## 1. TC-Z6 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z6'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z6 결측치 발생 시점 및 LOT 확인
- `TC-Z6` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z6'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z6 온도 집계
- 작업 단위(LOT)별 6존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z6'

# LOT별 TC-Z6 온도 집계
tc6_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc6_lot_summary['평균온도'] = tc6_lot_summary['평균온도'].round(2)

print(f"총 {len(tc6_lot_summary)}개 LOT에 대한 TC-Z6 수치 요약:\n")
for i, row in tc6_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z6 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


### <span id="desc-op-z3"></span>`OP-Z3` (가열·균열존 1존 제어출력)

## 1. OP-Z3 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 0% 출력(Zero) 비율, 평균, 최소/최대 제어출력 수치를 확인합니다.

In [ ]:
col = 'OP-Z3'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
zero_rate = (zero_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 0% 출력(Zero) 건수 : {zero_cnt:,}건 ({zero_rate:.2f}%)")
print("-" * 40)
print(f"· 평균 제어출력 (Mean) : {stats['mean']:.2f} %")
print(f"· 표준편차 (Std)       : {stats['std']:.2f} %")
print(f"· 최소 제어출력 (Min)  : {stats['min']:.2f} %")
print(f"· 중앙값 (Median)      : {stats['50%']:.2f} %")
print(f"· 최대 제어출력 (Max)  : {stats['max']:.2f} %")

## 2. OP-Z3 결측 발생 시각

상세 행은 `data/processed/IWEasdwe1234/OP-Z3_결측치_상세.csv`에 저장함.


In [ ]:
col = 'OP-Z3'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")

if len(null_rows) > 0:
    # 상위 10개 샘플 확인
    print("\n[상위 10개 결측 행 샘플]")
    display(null_rows[['MEAS_DT', 'LOT_NO', col]].head(10))
    
    # 결측치 상세 데이터를 외부 CSV 파일로 내보내기
    output_dir = os.path.join(PROCESSED_DIR, "IWEasdwe1234")
    os.makedirs(output_dir, exist_ok=True)
    csv_output_path = os.path.join(output_dir, "OP-Z3_결측치_상세.csv")
    null_rows.to_csv(csv_output_path, index=False, encoding=ENC)
    print(f"\n✅ 전체 {len(null_rows)}건의 결측치 데이터 저장 완료 -> {csv_output_path}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 OP-Z3 제어출력 집계
- 작업 단위(LOT)별 건조/가열 3존 제어출력(%)의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'OP-Z3'

# LOT별 OP-Z3 제어출력 집계
op3_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균출력='mean',
    최저출력='min',
    최고출력='max'
).reset_index()

# 소수점 둘째 자리 맞춤
op3_lot_summary['평균출력'] = op3_lot_summary['평균출력'].round(2)

print(f"총 {len(op3_lot_summary)}개 LOT에 대한 OP-Z3 수치 요약:\n")
for i, row in op3_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['row' if 'row' in row else 'LOT_NO']} | 평균: {row['평균출력']:6.2f}% | 범위: {row['최저출력']:6.2f}% ~ {row['최고출력']:6.2f}% ({row['측정건수']:,}건)")

### OP-Z3 읽는 방법

결측 254건을 별도 CSV로 저장함. 출력 0은 결측으로 바꾸지 않음. 출력이 0인 이유나 고장 여부는 같은 시간의 온도와 정비기록을 함께 보고 판단함.


### <span id="desc-cp-z4m"></span>`CP-Z4M` (카본포텐셜 모니터)

## 1. CP-Z4M 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 비율, 평균, 최소/최대 측정 수치를 확인합니다.

In [ ]:
col = 'CP-Z4M'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
zero_rate = (zero_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt:,}건 ({zero_rate:.2f}%)")
print("-" * 40)
print(f"· 평균 측정값 (Mean) : {stats['mean']:.3e}")
print(f"· 표준편차 (Std)     : {stats['std']:.3e}")
print(f"· 최소 측정값 (Min)  : {stats['min']:.3e}")
print(f"· 중앙값 (Median)    : {stats['50%']:.3e}")
print(f"· 최대 측정값 (Max)  : {stats['max']:.3e}")

## 2. CP-Z4M 결측치 발생 시점 및 LOT 확인
- `CP-Z4M` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'CP-Z4M'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 CP-Z4M 수치 집계
- 작업 단위(LOT)별 카본포텐셜 모니터 수치(`CP-Z4M`)의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'CP-Z4M'

# LOT별 CP-Z4M 수치 집계
cp4m_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균값='mean',
    최저값='min',
    최고값='max'
).reset_index()

# 작은 값을 0으로 반올림하지 않음.

print(f"총 {len(cp4m_lot_summary)}개 LOT에 대한 CP-Z4M 수치 요약:\n")
for i, row in cp4m_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균값']:.3e} | 범위: {row['최저값']:.3e} ~ {row['최고값']:.3e} ({row['측정건수']:,}건)")

### CP-Z4M 읽는 방법

값이 약 `10⁻¹¹~10⁻¹⁰`이라 소수점 두 자리로 표시하면 0처럼 보임. 지수 표기로 표시하며 실제 0은 없음.
데이터사전은 `TC-Z5`와 거의 중복된 신호라고 설명함. `CP-Z4`의 피드백이나 가스 차단 신호로 해석하지 않음. 단위는 확인되지 않음.


### <span id="desc-tc-z7"></span>`TC-Z7` (과시효대 1존 온도)

## 1. TC-Z7 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z7'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z7 결측치 발생 시점 및 LOT 확인
- `TC-Z7` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z7'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z7 온도 집계
- 작업 단위(LOT)별 과시효대 1존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z7'

# LOT별 TC-Z7 온도 집계
tc7_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc7_lot_summary['평균온도'] = tc7_lot_summary['평균온도'].round(2)

print(f"총 {len(tc7_lot_summary)}개 LOT에 대한 TC-Z7 수치 요약:\n")
for i, row in tc7_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z7 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-z8"></span>`TC-Z8` (과시효대 2존 온도)

## 1. TC-Z8 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z8'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z8 결측치 발생 시점 및 LOT 확인
- `TC-Z8` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z8'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z8 온도 집계
- 작업 단위(LOT)별 과시효대 2존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z8'

# LOT별 TC-Z8 온도 집계
tc8_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc8_lot_summary['평균온도'] = tc8_lot_summary['평균온도'].round(2)

print(f"총 {len(tc8_lot_summary)}개 LOT에 대한 TC-Z8 수치 요약:\n")
for i, row in tc8_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z8 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-z9"></span>`TC-Z9` (과시효대 슬러지 제거부 온도)

## 1. TC-Z9 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-Z9'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-Z9 결측치 발생 시점 및 LOT 확인
- `TC-Z9` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-Z9'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-Z9 온도 집계
- 작업 단위(LOT)별 과시효대 3존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-Z9'

# LOT별 TC-Z9 온도 집계
tc9_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tc9_lot_summary['평균온도'] = tc9_lot_summary['평균온도'].round(2)

print(f"총 {len(tc9_lot_summary)}개 LOT에 대한 TC-Z9 수치 요약:\n")
for i, row in tc9_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-Z9 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-op-z7"></span>`OP-Z7` (과시효대 1존 제어출력)

## 1. OP-Z7 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 비율, 평균, 최소/최대 조작량 수치를 확인합니다.

In [ ]:
col = 'OP-Z7'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
zero_rate = (zero_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt:,}건 ({zero_rate:.2f}%)")
print("-" * 40)
print(f"· 평균 조작량 (Mean) : {stats['mean']:.2f} %")
print(f"· 표준편차 (Std)     : {stats['std']:.2f} %")
print(f"· 최소 조작량 (Min)  : {stats['min']:.2f} %")
print(f"· 중앙값 (Median)    : {stats['50%']:.2f} %")
print(f"· 최대 조작량 (Max)  : {stats['max']:.2f} %")

## 2. OP-Z7 결측치 발생 시점 및 LOT 확인
- `OP-Z7` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'OP-Z7'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 OP-Z7 조작량 집계
- 작업 단위(LOT)별 과시효대 1존 조작량(`OP-Z7`)의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'OP-Z7'

# LOT별 OP-Z7 조작량 집계
op7_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균조작량='mean',
    최저조작량='min',
    최고조작량='max'
).reset_index()

# 소수점 둘째 자리 맞춤
op7_lot_summary['평균조작량'] = op7_lot_summary['평균조작량'].round(2)

print(f"총 {len(op7_lot_summary)}개 LOT에 대한 OP-Z7 수치 요약:\n")
for i, row in op7_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균조작량']:6.2f}% | 범위: {row['최저조작량']:6.2f}% ~ {row['최고조작량']:6.2f}% ({row['측정건수']:,}건)")

### OP-Z7 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-op-z8"></span>`OP-Z8` (과시효대 2존 제어출력)

## 1. OP-Z8 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 비율, 평균, 최소/최대 조작량 수치를 확인합니다.

In [ ]:
col = 'OP-Z8'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
zero_rate = (zero_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt:,}건 ({zero_rate:.2f}%)")
print("-" * 40)
print(f"· 평균 조작량 (Mean) : {stats['mean']:.2f} %")
print(f"· 표준편차 (Std)     : {stats['std']:.2f} %")
print(f"· 최소 조작량 (Min)  : {stats['min']:.2f} %")
print(f"· 중앙값 (Median)    : {stats['50%']:.2f} %")
print(f"· 최대 조작량 (Max)  : {stats['max']:.2f} %")

## 2. OP-Z8 결측치 발생 시점 및 LOT 확인
- `OP-Z8` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'OP-Z8'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 OP-Z8 조작량 집계
- 작업 단위(LOT)별 과시효대 2존 조작량(`OP-Z8`)의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'OP-Z8'

# LOT별 OP-Z8 조작량 집계
op8_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균조작량='mean',
    최저조작량='min',
    최고조작량='max'
).reset_index()

# 소수점 둘째 자리 맞춤
op8_lot_summary['평균조작량'] = op8_lot_summary['평균조작량'].round(2)

print(f"총 {len(op8_lot_summary)}개 LOT에 대한 OP-Z8 수치 요약:\n")
for i, row in op8_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균조작량']:6.2f}% | 범위: {row['최저조작량']:6.2f}% ~ {row['최고조작량']:6.2f}% ({row['측정건수']:,}건)")

### OP-Z8 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-out1"></span>`TC-OUT1` (냉각대 1존 온도)

## 1. TC-OUT1 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-OUT1'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-OUT1 결측치 발생 시점 및 LOT 확인
- `TC-OUT1` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-OUT1'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-OUT1 온도 집계
- 작업 단위(LOT)별 추출측 1존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-OUT1'

# LOT별 TC-OUT1 온도 집계
tcout1_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tcout1_lot_summary['평균온도'] = tcout1_lot_summary['평균온도'].round(2)

print(f"총 {len(tcout1_lot_summary)}개 LOT에 대한 TC-OUT1 수치 요약:\n")
for i, row in tcout1_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-OUT1 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.


# <span id="desc-tc-out2"></span>`TC-OUT2` (냉각대 2존 온도)

## 1. TC-OUT2 컬럼 기본 수치 통계량
- 전체 데이터 건수, 결측치, 수치 0 존재 여부, 평균, 최소/최대 온도 수치를 확인합니다.

In [ ]:
col = 'TC-OUT2'

total_cnt = len(df)
null_cnt = df[col].isnull().sum()
zero_cnt = (df[col] == 0).sum()
null_rate = (null_cnt / total_cnt) * 100
valid_cnt = df[col].count()

# 기술 통계량 산출
stats = df[col].describe()

print(f"=== [{col}] 컬럼 데이터 요약 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 유효 측정 데이터 : {valid_cnt:,}건")
print(f"· 결측치(Null) 건수 : {null_cnt}건 ({null_rate:.4f}%)")
print(f"· 수치 0(Zero) 건수 : {zero_cnt}건 (정상 가동 중 0°C 없음)")
print("-" * 40)
print(f"· 평균 온도 (Mean) : {stats['mean']:.2f} °C")
print(f"· 표준편차 (Std)   : {stats['std']:.2f} °C")
print(f"· 최소 온도 (Min)  : {stats['min']:.2f} °C")
print(f"· 중앙값 (Median)  : {stats['50%']:.2f} °C")
print(f"· 최대 온도 (Max)  : {stats['max']:.2f} °C")

## 2. TC-OUT2 결측치 발생 시점 및 LOT 확인
- `TC-OUT2` 컬럼이 비어있는(Null) 행의 측정 시각과 작업 LOT 번호를 확인합니다.

In [ ]:
col = 'TC-OUT2'
null_rows = df[df[col].isnull()]

print(f"=== [{col}] 결측치 발생 상세 ({len(null_rows)}건) ===")
if len(null_rows) > 0:
    for i, (idx, row) in enumerate(null_rows.iterrows(), 1):
        print(f"{i:2d}. [행 번호: {idx:>6d}] 시각: {row['MEAS_DT']} | LOT_NO: {row['LOT_NO']}")
else:
    print("결측치가 존재하지 않습니다.")

## 3. LOT_NO별 TC-OUT2 온도 집계
- 작업 단위(LOT)별 추출측 2존 온도의 평균, 최소, 최대 수치를 비교합니다.

In [ ]:
col = 'TC-OUT2'

# LOT별 TC-OUT2 온도 집계
tcout2_lot_summary = df.groupby('LOT_NO')[col].agg(
    측정건수='count',
    평균온도='mean',
    최저온도='min',
    최고온도='max'
).reset_index()

# 소수점 둘째 자리 맞춤
tcout2_lot_summary['평균온도'] = tcout2_lot_summary['평균온도'].round(2)

print(f"총 {len(tcout2_lot_summary)}개 LOT에 대한 TC-OUT2 수치 요약:\n")
for i, row in tcout2_lot_summary.iterrows():
    print(f"{i+1:2d}. LOT {row['LOT_NO']} | 평균: {row['평균온도']:6.2f}°C | 범위: {row['최저온도']:6.2f}°C ~ {row['최고온도']:6.2f}°C ({row['측정건수']:,}건)")

### TC-OUT2 확인 결과

위 표에서 결측 시각과 LOT별 값 범위를 확인함. 결측 발생 원인은 제공 기록만으로 확정하지 않음.
